# ADFTD EEG preprocessing for SDN experiments

This notebook prepares the ADFTD EEG input files used in the SDN bio-signal classification experiments.

The original OpenNeuro ds004504 dataset is distributed in a BIDS-style directory structure with subject folders such as `sub-001`, `sub-002`, and a `participants.tsv` file. This notebook first reorganizes the downloaded OpenNeuro files into class-wise folders using the diagnostic group information in `participants.tsv`. The subsequent preprocessing steps operate on the reorganized working directory.

The processing performed here is intentionally minimal:

1. Reorganize OpenNeuro `.set` files into class-wise folders.
2. Convert EEGLAB `.set` files to `.csv` files.
3. Segment each CSV recording into non-overlapping 20-sample windows.
4. Save segment paths and labels using a stratified train/test split.

No additional filtering, denoising, artifact removal, normalization, or handcrafted feature extraction is applied in this repository. If the source-preprocessed files under the OpenNeuro `derivatives/` directory are used, those preprocessing steps were performed by the original dataset authors, not by this notebook.

## 0. Directory structure

### Downloaded OpenNeuro directory

Set `OPENNEURO_ROOT` to the downloaded OpenNeuro ds004504 root directory. It should look like this:

```text
ds004504/
├── CHANGES
├── README
├── dataset_description.json
├── participants.tsv
├── derivatives/
├── sub-001/
├── sub-002/
├── sub-003/
└── ...
```

### Working directory generated by this notebook

Set `ROOT_DIR` to a separate local workspace. After the reorganization step, the notebook creates the following class-wise structure:

```text
adftd_eeg/
├── set_files/
│   ├── class_A/
│   ├── class_C/
│   └── class_F/
├── raw_csv/
│   ├── class_A/
│   ├── class_C/
│   └── class_F/
├── raw_npy_20_nooverlap/
│   ├── class_A/
│   ├── class_C/
│   └── class_F/
└── splits/
```

Class mapping:

- `class_A`: Alzheimer's disease (AD)
- `class_C`: cognitively normal/control (CN)
- `class_F`: frontotemporal dementia (FTD)

In [ ]:
from pathlib import Path
import re
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

try:
    import mne
except ImportError:
    mne = None

In [ ]:
# ---------------------------------------------------------------------
# User configuration
# ---------------------------------------------------------------------
ROOT_DIR = Path('/path/to/adftd_eeg')  # Change this to your local ADFTD EEG directory.

SET_INPUT_DIR = ROOT_DIR / 'set_files'
CSV_OUTPUT_DIR = ROOT_DIR / 'raw_csv'
SEGMENT_OUTPUT_DIR = ROOT_DIR / 'raw_npy_20_nooverlap'
SPLIT_OUTPUT_DIR = ROOT_DIR / 'splits'

SEGMENT_METADATA_PATH = ROOT_DIR / 'segment_metadata_20.csv'

CLASS_MAP: Dict[str, int] = {
    'class_A': 0,  # Alzheimer's disease
    'class_C': 1,  # cognitively normal/control
    'class_F': 2,  # frontotemporal dementia
}

LABEL_NAMES: Dict[int, str] = {
    0: 'AD',
    1: 'CN',
    2: 'FTD',
}

# 19 standard 10-20 EEG channels used in the manuscript.
EXPECTED_CHANNELS: List[str] = [
    'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz'
]

N_CHANNELS = 19
WINDOW_SIZE = 20
STEP_SIZE = 20
CHANNELS_FIRST = True  # True -> saved segment shape is (19, 20). False -> (20, 19).

TEST_SIZE = 0.2
RANDOM_STATE = 42

In [ ]:
import shutil

# ---------------------------------------------------------------------
# Optional: OpenNeuro BIDS root -> class-wise working directory
# ---------------------------------------------------------------------
# Set this to the downloaded OpenNeuro ds004504 root directory, i.e.,
# the folder containing participants.tsv and sub-001, sub-002, ... folders.
OPENNEURO_ROOT = Path('/path/to/ds004504')

# Set True only if you want to use .set files under the OpenNeuro derivatives/ folder.
# Set False to use .set files under the original sub-* folders.
USE_DERIVATIVES = False


def _normalize_subject_id(subject_id: object) -> str:
    subject_id = str(subject_id).strip()
    if subject_id.startswith('sub-'):
        return subject_id
    if subject_id.isdigit():
        return f'sub-{int(subject_id):03d}'
    return subject_id


def _map_group_to_class(group_value: object) -> str:
    group = str(group_value).strip().lower()
    group = group.replace('_', ' ').replace('-', ' ')
    group = ' '.join(group.split())

    if group in {'a', 'ad', 'alzheimer', 'alzheimers disease', "alzheimer's disease"}:
        return 'class_A'
    if group in {'c', 'cn', 'control', 'healthy', 'healthy control', 'healthy subjects', 'healthy subject', 'normal'}:
        return 'class_C'
    if group in {'f', 'ftd', 'frontotemporal dementia'}:
        return 'class_F'

    raise ValueError(f'Unknown diagnostic group label: {group_value}')


def _find_group_column(participants: pd.DataFrame) -> str:
    candidates = [
        'Group', 'group', 'diagnosis', 'Diagnosis', 'diagnostic_group',
        'participant_group', 'condition', 'Condition'
    ]
    for column in candidates:
        if column in participants.columns:
            return column
    raise ValueError(
        'Could not find a group/diagnosis column in participants.tsv. '
        f'Available columns: {participants.columns.tolist()}'
    )


def load_subject_to_class(openneuro_root: Path = OPENNEURO_ROOT) -> Dict[str, str]:
    """Read participants.tsv and return a mapping from subject ID to class folder."""
    participants_path = openneuro_root / 'participants.tsv'
    if not participants_path.exists():
        raise FileNotFoundError(f'Cannot find participants.tsv at: {participants_path}')

    participants = pd.read_csv(participants_path, sep='\t')
    if 'participant_id' not in participants.columns:
        raise ValueError("participants.tsv must contain a 'participant_id' column.")

    group_column = _find_group_column(participants)

    subject_to_class: Dict[str, str] = {}
    for _, row in participants.iterrows():
        subject_id = _normalize_subject_id(row['participant_id'])
        subject_to_class[subject_id] = _map_group_to_class(row[group_column])

    return subject_to_class


def reorganize_openneuro_to_class_folders(
    openneuro_root: Path = OPENNEURO_ROOT,
    set_input_dir: Path = SET_INPUT_DIR,
    use_derivatives: bool = USE_DERIVATIVES,
    overwrite: bool = False,
) -> pd.DataFrame:
    """
    Copy OpenNeuro EEGLAB .set files into set_files/class_A, class_C, and class_F.

    This function does not modify the original OpenNeuro dataset. It only copies files
    into the working directory used by the rest of this notebook. If paired .fdt files
    exist, they are copied together with the corresponding .set files.
    """
    subject_to_class = load_subject_to_class(openneuro_root)

    for class_name in CLASS_MAP:
        ensure_dir(set_input_dir / class_name)

    if use_derivatives:
        search_roots = [openneuro_root / 'derivatives']
    else:
        search_roots = [path for path in sorted(openneuro_root.glob('sub-*')) if path.is_dir()]

    set_files: List[Path] = []
    for search_root in search_roots:
        if search_root.exists():
            set_files.extend(sorted(search_root.rglob('*.set')))

    if not set_files:
        raise FileNotFoundError(f'No .set files found under: {search_roots}')

    rows: List[dict] = []
    for set_path in set_files:
        subject_id = next((part for part in set_path.parts if part.startswith('sub-')), None)
        if subject_id is None:
            print(f'[WARN] Could not identify subject ID from path: {set_path}')
            continue
        if subject_id not in subject_to_class:
            print(f'[WARN] {subject_id} not found in participants.tsv. Skipping: {set_path.name}')
            continue

        class_name = subject_to_class[subject_id]
        target_set_path = set_input_dir / class_name / set_path.name

        if overwrite or not target_set_path.exists():
            shutil.copy2(set_path, target_set_path)

        source_fdt_path = set_path.with_suffix('.fdt')
        target_fdt_path = target_set_path.with_suffix('.fdt')
        copied_fdt = False
        if source_fdt_path.exists():
            if overwrite or not target_fdt_path.exists():
                shutil.copy2(source_fdt_path, target_fdt_path)
            copied_fdt = True

        rows.append({
            'subject_id': subject_id,
            'class_folder': class_name,
            'source_set': str(set_path),
            'target_set': str(target_set_path),
            'paired_fdt_copied': copied_fdt,
        })

    summary = pd.DataFrame(rows)
    print(f'Copied or verified {len(summary)} .set files into: {set_input_dir}')
    if not summary.empty:
        print(summary.groupby('class_folder').size().reset_index(name='n_set_files'))

    return summary


# Run this once before convert_all_set_to_csv() if starting from the original OpenNeuro structure.
# reorganize_summary = reorganize_openneuro_to_class_folders()

## 1. Convert `.set` files to `.csv`

This step writes one CSV file per EEG recording. Rows are time samples and columns are EEG channels. No signal processing is applied during conversion.


In [ ]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def list_files_by_class(base_dir: Path, suffix: str) -> List[Tuple[str, Path]]:
    files: List[Tuple[str, Path]] = []
    for class_name in CLASS_MAP:
        class_dir = base_dir / class_name
        if not class_dir.exists():
            print(f'[WARN] Missing directory: {class_dir}')
            continue
        files.extend((class_name, path) for path in sorted(class_dir.rglob(f'*{suffix}')))
    return files


def relative_posix(path: Path, base_dir: Path = ROOT_DIR) -> str:
    try:
        return path.resolve().relative_to(base_dir.resolve()).as_posix()
    except ValueError:
        return path.as_posix()


def convert_one_set_to_csv(set_path: Path, csv_path: Path) -> None:
    if mne is None:
        raise ImportError('MNE is required for .set conversion. Install it with: pip install mne')

    raw = mne.io.read_raw_eeglab(str(set_path), preload=True, verbose='ERROR')

    if all(channel in raw.ch_names for channel in EXPECTED_CHANNELS):
        raw.pick_channels(EXPECTED_CHANNELS, ordered=True)
    else:
        missing = [channel for channel in EXPECTED_CHANNELS if channel not in raw.ch_names]
        raise ValueError(f'{set_path.name}: missing expected EEG channels: {missing}')

    eeg = raw.get_data().T  # time × channels
    df = pd.DataFrame(eeg, columns=raw.ch_names)

    ensure_dir(csv_path.parent)
    df.to_csv(csv_path, index=False)


def convert_all_set_to_csv(set_input_dir: Path = SET_INPUT_DIR, csv_output_dir: Path = CSV_OUTPUT_DIR) -> None:
    set_files = list_files_by_class(set_input_dir, '.set')
    print(f'Found {len(set_files)} .set files.')

    counts = {class_name: 0 for class_name in CLASS_MAP}
    for class_name, set_path in set_files:
        relative_path = set_path.relative_to(set_input_dir / class_name).with_suffix('.csv')
        csv_path = csv_output_dir / class_name / relative_path
        convert_one_set_to_csv(set_path, csv_path)
        counts[class_name] += 1

    print('Converted files by class:')
    for class_name, count in counts.items():
        print(f'  {class_name}: {count}')

In [ ]:
# Run this once before convert_all_set_to_csv() if starting from the original OpenNeuro structure.
reorganize_summary = reorganize_openneuro_to_class_folders()

## 2. Segment CSV files into 20-sample windows

Each CSV recording is divided into non-overlapping windows using `WINDOW_SIZE = 20` and `STEP_SIZE = 20`.
Each saved segment has shape `(19, 20)` when `CHANNELS_FIRST = True`.


In [ ]:
def clean_recording_name(csv_path: Path) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', csv_path.stem)


def read_eeg_csv(csv_path: Path) -> pd.DataFrame:
    """Read one EEG CSV file and return a numeric dataframe with 19 channels."""
    df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')

    if all(channel in df.columns for channel in EXPECTED_CHANNELS):
        df = df[EXPECTED_CHANNELS]
    else:
        # Fall back to headerless reading. This avoids losing the first sample
        # when CSV files were saved without column names.
        df = pd.read_csv(csv_path, header=None, engine='python', on_bad_lines='skip')
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna(axis=1, how='all')
        df = df.dropna(axis=0, how='any')
        if df.shape[1] > N_CHANNELS:
            raise ValueError(f'{csv_path} has {df.shape[1]} numeric channels; expected {N_CHANNELS}.')


    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.dropna(axis=0, how='any')

    if df.shape[1] != N_CHANNELS:
        raise ValueError(f'{csv_path} has {df.shape[1]} numeric channels; expected {N_CHANNELS}.')

    return df


def segment_one_csv(csv_path: Path, class_name: str, output_dir: Path = SEGMENT_OUTPUT_DIR) -> List[dict]:
    df = read_eeg_csv(csv_path)
    data = df.to_numpy(dtype=np.float32)  # time × channels

    recording_name = clean_recording_name(csv_path)
    rows: List[dict] = []

    for segment_index, start in enumerate(range(0, len(data) - WINDOW_SIZE + 1, STEP_SIZE)):
        end = start + WINDOW_SIZE
        segment = data[start:end, :]  # 20 × 19

        if CHANNELS_FIRST:
            segment = segment.T  # 19 × 20

        out_path = output_dir / class_name / f'{recording_name}_sample_{segment_index:06d}.npy'
        ensure_dir(out_path.parent)
        np.save(out_path, segment)

        rows.append({
            'segment_path': relative_posix(out_path),
            'source_csv': relative_posix(csv_path),
            'recording': recording_name,
            'class_folder': class_name,
            'label': CLASS_MAP[class_name],
            'label_name': LABEL_NAMES[CLASS_MAP[class_name]],
            'segment_index': segment_index,
            'start_sample': start,
            'end_sample': end,
        })

    return rows


def segment_all_csv(
    csv_dir: Path = CSV_OUTPUT_DIR,
    output_dir: Path = SEGMENT_OUTPUT_DIR,
    metadata_path: Path = SEGMENT_METADATA_PATH,
) -> pd.DataFrame:
    csv_files = list_files_by_class(csv_dir, '.csv')
    print(f'Found {len(csv_files)} CSV files.')

    all_rows: List[dict] = []
    for class_name, csv_path in csv_files:
        rows = segment_one_csv(csv_path, class_name, output_dir)
        all_rows.extend(rows)

    metadata = pd.DataFrame(all_rows)
    ensure_dir(metadata_path.parent)
    metadata.to_csv(metadata_path, index=False)

    print(f'Saved segment metadata to: {metadata_path}')
    if not metadata.empty:
        print('Generated segments by class:')
        print(metadata.groupby(['class_folder', 'label', 'label_name']).size().reset_index(name='n_segments'))

    return metadata

## 3. Split and save

This cell creates a manuscript-style stratified train/test split from the generated segment files.
Because the ADFTD segment set can be large, the notebook saves file paths and labels rather than loading all `.npy` arrays into memory.


In [ ]:
def split_and_save(
    metadata_path: Path = SEGMENT_METADATA_PATH,
    split_output_dir: Path = SPLIT_OUTPUT_DIR,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    if not metadata_path.exists():
        raise FileNotFoundError(f'Cannot find {metadata_path}. Run segment_all_csv() first.')

    metadata = pd.read_csv(metadata_path)
    required_columns = {'segment_path', 'label'}
    missing = required_columns.difference(metadata.columns)
    if missing:
        raise ValueError(f'Missing required columns in {metadata_path}: {sorted(missing)}')

    indices = np.arange(len(metadata))
    labels = metadata['label'].to_numpy(dtype=np.int64)

    train_index, test_index = train_test_split(
        indices,
        test_size=test_size,
        random_state=random_state,
        stratify=labels,
    )

    metadata['split'] = 'unused'
    metadata.loc[train_index, 'split'] = 'train'
    metadata.loc[test_index, 'split'] = 'test'

    train_metadata = metadata.loc[train_index].reset_index(drop=True)
    test_metadata = metadata.loc[test_index].reset_index(drop=True)

    train_files = train_metadata['segment_path'].to_numpy(dtype=str)
    test_files = test_metadata['segment_path'].to_numpy(dtype=str)
    y_train = train_metadata['label'].to_numpy(dtype=np.int64)
    y_test = test_metadata['label'].to_numpy(dtype=np.int64)

    ensure_dir(split_output_dir)

    np.save(split_output_dir / 'adftd_train_files_20.npy', train_files)
    np.save(split_output_dir / 'adftd_test_files_20.npy', test_files)
    np.save(split_output_dir / 'adftd_train_labels_20.npy', y_train)
    np.save(split_output_dir / 'adftd_test_labels_20.npy', y_test)

    np.savez_compressed(
        split_output_dir / 'adftd_20sample_train_test_files.npz',
        train_files=train_files,
        test_files=test_files,
        y_train=y_train,
        y_test=y_test,
        train_index=train_index,
        test_index=test_index,
    )

    split_metadata_path = split_output_dir / 'adftd_split_metadata_20.csv'
    metadata.to_csv(split_metadata_path, index=False)

    print(f'Saved split files to: {split_output_dir}')
    print(f'Train segments: {len(train_files):,}')
    print(f'Test segments:  {len(test_files):,}')
    print('Train label distribution:')
    print(pd.Series(y_train).value_counts().sort_index())
    print('Test label distribution:')
    print(pd.Series(y_test).value_counts().sort_index())

    return metadata